# Module 2 — Làm sạch và chuẩn hóa dữ liệu điện gió

Notebook đọc `T1.csv`, làm sạch, chuẩn hóa rồi lưu ra `Data/output/du_lieu_sach.csv`.

**Nguyên tắc quan trọng:** không xóa hẳn dữ liệu lỗi. Giá trị ngoài ngưỡng vật lý được thay bằng `NaN` rồi **nội suy tuyến tính có kiểm soát** (tối đa 6 bước liên tiếp = 1 giờ). Nếu mất tín hiệu quá dài (> 6 bước) thì để nguyên `NaN`, tránh bịa số cho khoảng trống lớn.

## Bước 1 — Đọc dữ liệu và đổi tên cột sang dạng ngắn không dấu

In [1]:
import os
import pandas as pd
import numpy as np

CSV_INPUT = "../Data/input/T1.csv"
CSV_OUTPUT = "../Data/output/du_lieu_sach.csv"

# Đọc dữ liệu gốc
df = pd.read_csv(CSV_INPUT)

# Đổi tên cột sang dạng ngắn, không dấu (theo đúng thứ tự cột gốc)
df = df.rename(columns={
    "Date/Time": "thoi_gian",
    "LV ActivePower (kW)": "cong_suat_kw",
    "Wind Speed (m/s)": "toc_do_gio",
    "Theoretical_Power_Curve (KWh)": "cong_suat_ly_thuyet",
    "Wind Direction (\u00b0)": "huong_gio",
})

print("Tên cột sau khi đổi:", list(df.columns))
df.head()

Tên cột sau khi đổi: ['thoi_gian', 'cong_suat_kw', 'toc_do_gio', 'cong_suat_ly_thuyet', 'huong_gio']


,thoi_gian,cong_suat_kw,toc_do_gio,cong_suat_ly_thuyet,huong_gio
0,01 01 2018 00:00,380.047791,5.311336,416.328908,259.994904
1,01 01 2018 00:10,453.769196,5.672167,519.917511,268.641113
2,01 01 2018 00:20,306.376587,5.216037,390.900016,272.564789
3,01 01 2018 00:30,419.645904,5.659674,516.127569,271.258087
4,01 01 2018 00:40,380.650696,5.577941,491.702972,265.674286


## Bước 2 — Chuyển kiểu dữ liệu

- `thoi_gian` → datetime, định dạng gốc `dd MM yyyy HH:mm`.
- Các cột còn lại → kiểu số (giá trị không hợp lệ sẽ thành `NaN`).

In [2]:
# Chuyển cột thời gian sang datetime
df["thoi_gian"] = pd.to_datetime(df["thoi_gian"], format="%d %m %Y %H:%M")

# Chuyển các cột còn lại sang kiểu số; giá trị lỗi -> NaN
cot_so = ["cong_suat_kw", "toc_do_gio", "cong_suat_ly_thuyet", "huong_gio"]
for cot in cot_so:
    df[cot] = pd.to_numeric(df[cot], errors="coerce")

print("Kiểu dữ liệu từng cột:")
print(df.dtypes)

Kiểu dữ liệu từng cột:
thoi_gian              datetime64[us]
cong_suat_kw                  float64
toc_do_gio                    float64
cong_suat_ly_thuyet           float64
huong_gio                     float64
dtype: object


## Bước 3 — Khử trùng thời gian, sắp xếp và đặt thoi_gian làm chỉ mục

In [3]:
so_dong_ban_dau = len(df)

# Khử các dòng trùng mốc thời gian (giữ dòng đầu tiên)
df = df.drop_duplicates(subset="thoi_gian", keep="first")
so_dong_sau_khu_trung = len(df)

# Sắp xếp theo thời gian tăng dần rồi đặt thoi_gian làm chỉ mục
df = df.sort_values("thoi_gian").set_index("thoi_gian")

print(f"Số dòng trùng thời gian đã loại: {so_dong_ban_dau - so_dong_sau_khu_trung}")
print(f"Số dòng còn lại: {so_dong_sau_khu_trung}")
df.head()

Số dòng trùng thời gian đã loại: 0
Số dòng còn lại: 50530


,cong_suat_kw,toc_do_gio,cong_suat_ly_thuyet,huong_gio
thoi_gian,,,,
2018-01-01 00:00:00,380.047791,5.311336,416.328908,259.994904
2018-01-01 00:10:00,453.769196,5.672167,519.917511,268.641113
2018-01-01 00:20:00,306.376587,5.216037,390.900016,272.564789
2018-01-01 00:30:00,419.645904,5.659674,516.127569,271.258087
2018-01-01 00:40:00,380.650696,5.577941,491.702972,265.674286


## Bước 4 — Loại giá trị ngoài ngưỡng vật lý rồi nội suy có kiểm soát

Ngưỡng hợp lệ:
- Công suất: **0 – 3600 kW**
- Tốc độ gió: **0 – 30 m/s**
- Hướng gió: **0 – 360°**

Giá trị ngoài ngưỡng → `NaN`, sau đó **nội suy tuyến tính giới hạn 6 bước liên tiếp**. Khoảng trống dài hơn 6 bước được giữ nguyên `NaN`.

In [4]:
# Định nghĩa ngưỡng vật lý (min, max) cho từng cột
nguong = {
    "cong_suat_kw": (0, 3600),
    "toc_do_gio": (0, 30),
    "huong_gio": (0, 360),
}

# Thay giá trị ngoài ngưỡng bằng NaN
so_ngoai_nguong = {}
for cot, (thap, cao) in nguong.items():
    ngoai = ~df[cot].between(thap, cao)  # True nếu ngoài ngưỡng (NaN cũng tính là ngoài)
    so_ngoai_nguong[cot] = int((ngoai & df[cot].notna()).sum())
    df.loc[ngoai, cot] = np.nan

print("Số giá trị ngoài ngưỡng bị thay bằng NaN:")
for cot, n in so_ngoai_nguong.items():
    print(f"  - {cot}: {n}")

# Nội suy tuyến tính có kiểm soát: tối đa 6 bước liên tiếp, chỉ nội suy khoảng bên trong
cot_noi_suy = ["cong_suat_kw", "toc_do_gio", "cong_suat_ly_thuyet", "huong_gio"]
df[cot_noi_suy] = df[cot_noi_suy].interpolate(
    method="linear", limit=6, limit_area="inside"
)

print("\nSố NaN còn lại sau khi nội suy (khoảng mất tín hiệu quá dài, giữ nguyên):")
print(df[cot_noi_suy].isna().sum())

Số giá trị ngoài ngưỡng bị thay bằng NaN:
  - cong_suat_kw: 2938
  - toc_do_gio: 0
  - huong_gio: 0

Số NaN còn lại sau khi nội suy (khoảng mất tín hiệu quá dài, giữ nguyên):
cong_suat_kw           1695
toc_do_gio                0
cong_suat_ly_thuyet       0
huong_gio                 0
dtype: int64


In [5]:
# Bỏ các dòng vẫn còn thiếu ở HAI cột quan trọng nhất: công suất và tốc độ gió
df_sach = df.dropna(subset=["cong_suat_kw", "toc_do_gio"])
print(f"Số dòng bị loại do vẫn thiếu công suất/tốc độ gió: {len(df) - len(df_sach)}")

Số dòng bị loại do vẫn thiếu công suất/tốc độ gió: 1695


## Bước 5 — So sánh số dòng trước và sau khi làm sạch

In [6]:
print(f"Số dòng ban đầu       : {so_dong_ban_dau}")
print(f"Số dòng sau làm sạch  : {len(df_sach)}")
print(f"Tổng số dòng đã loại  : {so_dong_ban_dau - len(df_sach)} "
      f"({(so_dong_ban_dau - len(df_sach)) / so_dong_ban_dau * 100:.2f}%)")

Số dòng ban đầu       : 50530
Số dòng sau làm sạch  : 48835
Tổng số dòng đã loại  : 1695 (3.35%)


## Bước 6 — Lưu kết quả ra file du_lieu_sach.csv

In [7]:
# Tạo thư mục output nếu chưa có, rồi lưu (giữ cả cột chỉ mục thoi_gian)
os.makedirs(os.path.dirname(CSV_OUTPUT), exist_ok=True)
df_sach.to_csv(CSV_OUTPUT, index=True)

print(f"Đã lưu dữ liệu sạch ra: {CSV_OUTPUT}")
print(f"Kích thước: {df_sach.shape[0]} dòng x {df_sach.shape[1]} cột")
df_sach.head()

Đã lưu dữ liệu sạch ra: ../Data/output/du_lieu_sach.csv
Kích thước: 48835 dòng x 4 cột


,cong_suat_kw,toc_do_gio,cong_suat_ly_thuyet,huong_gio
thoi_gian,,,,
2018-01-01 00:00:00,380.047791,5.311336,416.328908,259.994904
2018-01-01 00:10:00,453.769196,5.672167,519.917511,268.641113
2018-01-01 00:20:00,306.376587,5.216037,390.900016,272.564789
2018-01-01 00:30:00,419.645904,5.659674,516.127569,271.258087
2018-01-01 00:40:00,380.650696,5.577941,491.702972,265.674286
